# COMP8325 — Data Preparation

This notebook loads the BODMAS dataset, aligns features with metadata, and builds multi-class labels for malware category classification.

**Data directory:** `COMP8325-Assignment2-Group9/data/`

### How to load the bodmas.npz file:
(Based on https://whyisyoung.github.io/BODMAS/)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ── Data directory ──────────────────────────────────────────────────────────
# Update DATA_DIR to point to wherever the three BODMAS files live on your
# machine.  The path below matches the project folder structure.

NPZ_PATH  = "data/bodmas.npz"
META_PATH = "data/bodmas_metadata.csv"
CAT_PATH  = "data/bodmas_malware_category.csv"

# ── Load feature vectors and binary labels ──────────────────────────────────
raw      = np.load(NPZ_PATH)
X        = raw["X"]   # feature matrix  (N × 2381)
y_binary = raw["y"]   # 0 = benign, 1 = malware

print(f"X shape   : {X.shape}")
print(f"y shape   : {y_binary.shape}")
print(f"Benign    : {(y_binary == 0).sum():,}")
print(f"Malicious : {(y_binary == 1).sum():,}")

X shape   : (134435, 2381)
y shape   : (134435,)
Benign    : 77,142
Malicious : 57,293


In [2]:
# ── Load metadata and category labels ───────────────────────────────────────
meta = pd.read_csv(META_PATH)
cats = pd.read_csv(CAT_PATH)

# Harmonise column name (metadata uses 'sha', category CSV uses 'sha256')
if "sha" in meta.columns and "sha256" not in meta.columns:
    meta = meta.rename(columns={"sha": "sha256"})

print("Metadata columns :", list(meta.columns))
print("Category columns :", list(cats.columns))
print(f"Metadata rows    : {len(meta):,}")
print(f"Category rows    : {len(cats):,}")

Metadata columns : ['sha256', 'timestamp', 'family']
Category columns : ['sha256', 'category']
Metadata rows    : 134,435
Category rows    : 57,293


In [3]:
# ── Build multi-class labels ─────────────────────────────────────────────────
# The i-th row of X corresponds to the i-th row of bodmas_metadata.csv.
# Benign samples (y_binary == 0) get the label 'benign'.
# Malicious samples get their category from bodmas_malware_category.csv
# (joined on sha256 hash).  Do NOT use the family column from metadata.

merged = meta.merge(cats, on="sha256", how="left")

y_category = np.where(
    y_binary == 0,
    "benign",
    merged["category"].astype(str).values,
)

print("Class distribution:")
print(pd.Series(y_category).value_counts().to_string())

Class distribution:
benign                77142
trojan                29972
worm                  16697
backdoor               7331
downloader             1031
ransomware              821
dropper                 715
informationstealer      448
virus                   192
pua                      29
cryptominer              20
p2p-worm                 16
exploit                  12
trojan-gamethief          6
rootkit                   3


In [4]:
# ── Persist prepared data for downstream notebooks ──────────────────────────
import joblib

out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

joblib.dump({"X": X, "y_binary": y_binary, "y_category": y_category},
            out_dir / "prepared_data.joblib")

print("Saved prepared_data.joblib to", out_dir)

Saved prepared_data.joblib to artifacts
